# Proyecto final

Modelar una juntura GaAs–Au como la siguiente:

```
      Au | GaAs | Au
         ↓      ↓
     contacto  contacto
       tipo      tipo
     Schottky   Óhmico
```

con el objetivo de calcular la corriente que se mide entre contactos para determinadas tensiones.

La descripción será unidimensional:

```
        x=0    GaAs    x=L
   Au  |////////////////|  Au
     Schottky         Óhmico
```

Queremos ver cómo se comportan los portadores de carga en el semiconductor.

## Ecuaciones de continuidad

$$
\partial_t n_e(x,t) = -\partial_x J_e(x,t) - R_e(x,t) + G_e(x,t)
$$

donde $n_e$ es la densidad de electrones, $J_e$ la densidad de corriente, $R_e$ la recombinación y $G_e$ la generación, con:

- $J_e(x,t) = \mu_e\,\partial_x \phi(x,t)\, n_e(x,t) - D_e\,\partial_x n_e(x,t)$

  donde $\mu_e$ es la movilidad electrónica y $D_e$ la constante de difusión de los portadores.

- $R_e(x,t) = \dfrac{n_e(x,t)}{\tau_e}$

  donde $\tau_e$ es el tiempo de vida.

La corriente estará dada por:

$$
j_e = -e J_e = -e\mu_e\,\partial_x \phi(x,t)\, n_e(x,t) + e D_e\,\partial_x n_e(x,t)
$$

y la densidad de carga por:

$$
\rho_e(x,t) = -e\, n_e(x,t)
$$

En resumen:

$$
\left[\begin{aligned}
\partial_t n_e &= -\partial_x J_e - R_e + G_e \\
J_e &= \mu_e\,\partial_x \phi\, n_e - D_e\,\partial_x n_e
\end{aligned}\right]
$$

$$
\Rightarrow \quad \partial_t n_e = -\mu_e\,\partial^2_{xx}\phi\; n_e - \mu_e\,\partial_x \phi\,\partial_x n_e + D_e\,\partial^2_{xx} n_e - R_e + G_e
$$

Y para huecos es todo análogo, con las constantes que correspondan.

(Las últimas igualdades suponen $\mu_e$ y $D_e$ constantes en $x$.)

## Ecuación de Poisson

La **ecuación de Poisson** que hay que resolver en el semiconductor es:

$$
-\partial_x\left[\varepsilon_r(x)\,\partial_x \phi(x)\right] = \frac{e}{\varepsilon_0}\left[n_h(x,t) - n_e(x,t) + C(x,t)\right]
$$

donde $C(x,t) = N_d^+(x,t) - N_a^-(x,t)$ es el perfil de impurezas.

### Condiciones de contorno

**Contacto tipo Schottky ($x = 0$):**

$$
-J_e(0,t) = v_e\left[n_e(0,t) - n_e^0(0)\right]
$$

donde $v_e$ es la velocidad de recombinación superficial y $n_e^0(0)$ la densidad de equilibrio en el contacto. El signo menos aparece porque $J_e$ es el flujo en $+x$ y en $x=0$ la normal saliente apunta hacia $-x$: si $n_e(0) > n_e^0(0)$, los electrones salen hacia el metal ($J_e(0) < 0$). Además,

$$
\phi(0,t) = \varphi_{Stky} + V_{App}
$$

con $\varphi_{Stky} = 0{,}7\ \mathrm{V}$ (Fermi-Pinning).

**Contacto Óhmico ($x = L$):**

$$
n_e(L,t) = n_e^0(L)
$$

$$
\phi(L,t) = \varphi_{bi}(L)
$$

La tensión $V_{App}$ se aplica sólo en el contacto Schottky y el contacto óhmico queda como referencia. Si se sumara en ambos contactos, la diferencia de potencial entre ellos no dependería de $V_{App}$. El potencial de referencia en el contacto óhmico es

$$
\varphi_{bi}(L) = \frac{k_B T}{e}\,\ln\left[\frac{n_e^0(L)}{n_{isc}}\right]
$$

$$
n_e^0(L) = \frac{C(L)}{2} + \sqrt{\frac{C(L)^2}{4} + n_{isc}^2}
$$

(se toma la raíz positiva: con el signo $-$ la densidad resultaría negativa). La misma expresión con $C(x)$ define la densidad de equilibrio $n_e^0(x)$ en todo el dominio.

$$
n_{isc} = 2\left[\frac{2\pi k_B T}{h^2}\right]^{3/2}\left(m_e^*\, m_h^*\right)^{3/4} e^{-E_g / 2 k_B T}
$$

## Diferencias finitas

Llevamos el problema a algo resoluble computacionalmente, planteando las ecuaciones en **diferencias finitas**.

Para eso, discretizamos el tiempo como $t = m\,\Delta t$ (con $m = 0, \dots, M$ y $\Delta t = t_{max}/M$) y el espacio como $x = j\,\Delta x$ (con $j = 0, \dots, N$ y $\Delta x = L/N$).

Notación: $n_j^m \equiv n_e(x_j, t_m)$, $p_j^m \equiv n_h(x_j, t_m)$, $\phi_j^m \equiv \phi(x_j, t_m)$, $C_j^m \equiv C(x_j, t_m)$ y $G_j^m \equiv G_e(x_j, t_m)$.

### Ecuación de continuidad

Usando diferencias hacia adelante en el tiempo y centradas en el espacio:

$$
\begin{aligned}
n_j^{m+1} = n_j^m
&- \mu_e \frac{\Delta t}{\Delta x^2}\, n_j^m \left[\phi_{j+1}^m - 2\phi_j^m + \phi_{j-1}^m\right] \\
&- \mu_e \frac{\Delta t}{4\,\Delta x^2} \left[\phi_{j+1}^m - \phi_{j-1}^m\right]\left[n_{j+1}^m - n_{j-1}^m\right] \\
&+ D_e \frac{\Delta t}{\Delta x^2}\left[n_{j+1}^m - 2 n_j^m + n_{j-1}^m\right] \\
&- \Delta t\, \frac{n_j^m}{\tau_e} + \Delta t\, G_j^m
\end{aligned}
$$

para $j = 1, \dots, N-1$. Al ser un esquema explícito, requiere $D_e\,\Delta t/\Delta x^2 \le 1/2$ para ser estable. Además, si la deriva domina ($\mu_e |\Delta\phi| \gtrsim 2 D_e$ entre nodos vecinos), el centrado en el espacio puede oscilar; en ese caso conviene discretizar el flujo con el esquema de Scharfetter–Gummel.

### Ecuación de Poisson

Con $\varepsilon_r$ constante y diferencias centradas, para cada nodo interior $j = 1, \dots, N-1$:

$$
-\phi_{j-1}^m + 2\phi_j^m - \phi_{j+1}^m = \Delta x^2\, \frac{e}{\varepsilon_0 \varepsilon_r}\left[p_j^m - n_j^m + C_j^m\right]
$$

Es un problema de contorno: $\phi_0^m$ y $\phi_N^m$ están fijados por los contactos, así que no se puede avanzar nodo a nodo desde un extremo (no se impondría la condición del otro extremo y los errores se amplifican). Pasando los valores de borde al lado derecho queda un sistema tridiagonal de $N-1$ incógnitas:

$$
\begin{pmatrix}
2 & -1 & & \\
-1 & 2 & \ddots & \\
& \ddots & \ddots & -1 \\
& & -1 & 2
\end{pmatrix}
\begin{pmatrix} \phi_1^m \\ \phi_2^m \\ \vdots \\ \phi_{N-1}^m \end{pmatrix}
= \Delta x^2\, \frac{e}{\varepsilon_0 \varepsilon_r}
\begin{pmatrix} p_1^m - n_1^m + C_1^m \\ p_2^m - n_2^m + C_2^m \\ \vdots \\ p_{N-1}^m - n_{N-1}^m + C_{N-1}^m \end{pmatrix}
+ \begin{pmatrix} \phi_0^m \\ 0 \\ \vdots \\ \phi_N^m \end{pmatrix}
$$

que se resuelve en cada paso de tiempo con `scipy.linalg.solve_banded`.

### Condiciones de contorno

$$
\begin{aligned}
n_e(0,t) &\longrightarrow n_0^m \\
n_e(L,t) &\longrightarrow n_N^m \\
\phi(0,t) &\longrightarrow \phi_0^m \\
\phi(L,t) &\longrightarrow \phi_N^m
\end{aligned}
$$

### Condición inicial

Se parte de la densidad de equilibrio $n_e^0(x)$ (sin generación ni tensión aplicada):

$$
n_e(x,0) = n_e^0(x) \;\longrightarrow\; n_j^0 = n_e^0(x_j)
$$